# Damped Harmonic Oscillator: Interactive Comparison

Explore critical, overdamped, and underdamped motion by changing the initial conditions and damping coefficients. The notebook uses `ipywidgets` for controls and ordinary Matplotlib figure output; it does **not** require the interactive Matplotlib (`ipympl`) backend.

## Before you begin
Run the cells from top to bottom. If the controls below do not appear as sliders and a button, the notebook environment may not have the Jupyter widget frontend enabled.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

print('NumPy:', np.__version__)
print('Matplotlib:', plt.matplotlib.__version__)
print('ipywidgets:', widgets.__version__)
print('Matplotlib backend:', plt.get_backend())

NumPy: 2.4.6
Matplotlib: 3.10.9
ipywidgets: 7.8.5
Matplotlib backend: module://matplotlib_inline.backend_inline


## Model

We use the damped-oscillator equation

\\[\\frac{d^2x}{dt^2}+2\\gamma\\frac{dx}{dt}+\\omega_0^2x=0.\\]

For this demonstration, the natural angular frequency is fixed at \\(\\omega_0=1\\). The critical case therefore has \\(\\gamma=1\\). The initial position and velocity are shared by all three curves.

In [2]:
omega0 = 1.0
t = np.linspace(0.0, 12.0, 1201)

# Controls
position = widgets.FloatSlider(
    value=5.0, min=0.0, max=10.0, step=0.1,
    description='Initial position', readout_format='.1f',
    continuous_update=False, style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
velocity = widgets.FloatSlider(
    value=0.0, min=-10.0, max=10.0, step=0.1,
    description='Initial velocity', readout_format='.1f',
    continuous_update=False, style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
gamma_over = widgets.FloatSlider(
    value=1.5, min=1.01, max=5.0, step=0.01,
    description='Overdamped γ', readout_format='.2f',
    continuous_update=False, style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
gamma_under = widgets.FloatSlider(
    value=0.5, min=0.0, max=0.99, step=0.01,
    description='Underdamped γ', readout_format='.2f',
    continuous_update=False, style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
reset_button = widgets.Button(description='Reset values', button_style='')
plot_output = widgets.Output()

def oscillator_curves(x0, v0, gamma_over_value, gamma_under_value):
    # Critical damping: gamma = omega0
    gamma_c = omega0
    critical = (x0 + (v0 + gamma_c * x0) * t) * np.exp(-gamma_c * t)

    # Overdamped: gamma > omega0
    q = np.sqrt(gamma_over_value**2 - omega0**2)
    a1 = (v0 + (gamma_over_value + q) * x0) / (2.0 * q)
    a2 = ((q - gamma_over_value) * x0 - v0) / (2.0 * q)
    overdamped = a1 * np.exp((-gamma_over_value + q) * t) + a2 * np.exp((-gamma_over_value - q) * t)

    # Underdamped: gamma < omega0; this form avoids phase-angle division issues.
    wd = np.sqrt(omega0**2 - gamma_under_value**2)
    underdamped = np.exp(-gamma_under_value * t) * (
        x0 * np.cos(wd * t) + ((v0 + gamma_under_value * x0) / wd) * np.sin(wd * t)
    )
    return critical, overdamped, underdamped

def redraw(change=None):
    critical, overdamped, underdamped = oscillator_curves(
        position.value, velocity.value, gamma_over.value, gamma_under.value
    )
    with plot_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(t, critical, color='blue', linewidth=2, label='Critical damping (γ = 1)')
        ax.plot(t, overdamped, color='red', linewidth=2, label=f'Overdamped (γ = {gamma_over.value:.2f})')
        ax.plot(t, underdamped, color='green', linewidth=2, label=f'Underdamped (γ = {gamma_under.value:.2f})')
        ax.set(xlim=(0, 12), ylim=(-10, 10), xlabel='Time (arbitrary units)', ylabel='Position (arbitrary units)', title='Damped harmonic oscillator')
        ax.axhline(0, color='black', linewidth=0.7)
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()
        plt.show()
        plt.close(fig)

def reset_values(button):
    position.value = 5.0
    velocity.value = 0.0
    gamma_over.value = 1.5
    gamma_under.value = 0.5

for control in (position, velocity, gamma_over, gamma_under):
    control.observe(redraw, names='value')
reset_button.on_click(reset_values)

display(widgets.VBox([
    position, velocity, gamma_over, gamma_under,
    reset_button, plot_output
]))
redraw()

## Questions to investigate

1. Which curve crosses equilibrium more than once?
2. How does increasing the overdamped damping coefficient change the time required to return near equilibrium?
3. How does the underdamped curve change as \\(\\gamma\\) approaches the critical value of 1?
4. Change the initial velocity while keeping the initial position fixed. Which features of the curves change?

## Technical note

The sliders are `ipywidgets` controls. Every change recalculates the curves and redraws a standard Matplotlib figure. This avoids relying on Matplotlib's interactive canvas backend, although the Anaconda Notebooks frontend must still support standard Jupyter widgets.